In [0]:
# Parameters - supplied by the Lakeflow Job via base_parameters
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_name = dbutils.widgets.get("model_name")
n_estimators = int(dbutils.widgets.get("n_estimators"))
max_depth = int(dbutils.widgets.get("max_depth"))
experiment_name = dbutils.widgets.get("experiment_name")

# Inter-task communication: get training table from upstream generate_data task
full_training_table = dbutils.jobs.taskValues.get(taskKey="generate_data", key="full_table_name")

full_model_name = f"{catalog}.{schema}.{model_name}"

print(f"Model: {full_model_name}")
print(f"Training table (from generate_data task): {full_training_table}")
print(f"Hyperparams: n_estimators={n_estimators}, max_depth={max_depth}")

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Set MLflow to use Unity Catalog
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_name)

# Load training data
df = spark.read.table(full_training_table).toPandas()

# Split features and label
feature_cols = [c for c in df.columns if c.startswith("feature_")]
X = df[feature_cols]
y = df["label"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model and log to MLflow
with mlflow.start_run(run_name="rf_training") as run:
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Log parameters
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("training_table", full_training_table)

    # Log metrics
    accuracy = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metric("accuracy", accuracy)

    # Log and register model
    result = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name=full_model_name,
        input_example=X_test[:5]
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Model registered as: {full_model_name}")

In [0]:
import json
from mlflow import MlflowClient

client = MlflowClient()

# Get the latest version of the registered model
versions = client.search_model_versions(f"name='{full_model_name}'")
latest_version = max(versions, key=lambda v: int(v.version)).version

# Set Champion alias
client.set_registered_model_alias(full_model_name, "Champion", int(latest_version))
print(f"Set alias 'Champion' on {full_model_name} version {latest_version}")

# Set task values for downstream tasks (batch_inference)
dbutils.jobs.taskValues.set(key="full_model_name", value=full_model_name)
dbutils.jobs.taskValues.set(key="model_version", value=latest_version)
dbutils.jobs.taskValues.set(key="accuracy", value=round(accuracy, 4))
dbutils.jobs.taskValues.set(key="run_id", value=run.info.run_id)

print(f"\nTask values set:")
print(f"  full_model_name = {full_model_name}")
print(f"  model_version = {latest_version}")
print(f"  accuracy = {round(accuracy, 4)}")
print(f"  run_id = {run.info.run_id}")

# Exit with structured output for the orchestrator
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "run_id": run.info.run_id,
    "model": full_model_name,
    "version": latest_version,
    "accuracy": round(accuracy, 4)
}))